# Medical Chatbot with Nutritional RAG
### Disease Prediction -> Personalised Nutrition (up/down vs healthy adult normal)

**Pipeline Overview:**
1. Load 3 PDFs: `medic_book.pdf` (diseases), `Herbal_Nutrients_and_Their_Health_Benefits.pdf` (herbal), `nutrition.pdf` (nutrition)
2. Embed & store chunks in Pinecone with **source tags** for each book
3. **Step 0** -- Symptom extractor: LLM parses raw query -> structured JSON (symptoms, duration, temperature, age, onset, worsening, existing conditions)
4. **Step 1** -- RAG Chain A: disease identification using `symptom_context` + retrieved chunks
5. **Step 2** -- Fallback chain: pure LLM reasoning (no RAG) if Chain A returns `general`
6. **Step 3** -- Chain B: nutrition chain with richer retrieval query + `disease_info` -> 7-row up/down table
7. **Step 4** -- Severity triage: structured rules (age/temp/duration/worsening) + keyword match + LLM -> worst wins
8. Final response: disease explanation + personalised nutrition table + severity alert

## Step 1: Setup & Imports

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
GROQ_API_KEY     = os.getenv("GROQ_API_KEY")

os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["GROQ_API_KEY"]     = GROQ_API_KEY

# Optional: set Tesseract path if needed (Mac/conda)
# import pytesseract
# pytesseract.pytesseract.tesseract_cmd = "/usr/local/bin/tesseract"

print("✅ Environment loaded")


## Step 2: Load PDFs with Source Labels

Each document gets a `book_type` metadata tag so the retriever knows **which book** a chunk came from:
- `disease` → medic_book.pdf
- `herbal`  → Herbal_Nutrients_and_Their_Health_Benefits.pdf
- `nutrition` → nutrition.pdf

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
from typing import List

# ── CONFIG: update this path to wherever your PDFs live ──
DATA_DIR = "/Users/ritabratabhattacharya/Desktop/cd/Python/PROJECTS/Medical_Chatbot/data"

PDF_BOOK_MAP = {
    "medic_book.pdf":                               "disease",
    "Herbal_Nutrients_and_Their_Health_Benefits.pdf": "herbal",
    "nutrition.pdf":                                "nutrition",
}

def load_pdfs_with_tags(data_dir: str, book_map: dict) -> List[Document]:
    """Load PDFs and inject a 'book_type' metadata tag on every page."""
    all_docs = []
    for filename, book_type in book_map.items():
        filepath = os.path.join(data_dir, filename)
        if not os.path.exists(filepath):
            print(f"⚠️  File not found, skipping: {filepath}")
            continue
        try:
            loader = PyPDFLoader(filepath)
            pages  = loader.load()
            for page in pages:
                page.metadata["book_type"] = book_type
                page.metadata["source"]    = filename
            all_docs.extend(pages)
            print(f"✅ Loaded {len(pages):>4} pages  [{book_type}]  {filename}")
        except Exception as e:
            print(f"❌ Error loading {filename}: {e}")
    return all_docs

raw_docs = load_pdfs_with_tags(DATA_DIR, PDF_BOOK_MAP)
print(f"\nTotal pages loaded: {len(raw_docs)}")

## Step 3: Text Splitting

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(docs: List[Document], 
                    chunk_size: int = 600, 
                    chunk_overlap: int = 60) -> List[Document]:
    """
    Split documents into chunks, preserving all metadata (including book_type).
    Slightly larger chunks than original to retain more nutritional context.
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )
    return splitter.split_documents(docs)

text_chunks = split_documents(raw_docs)
print(f"Total chunks: {len(text_chunks)}")

# Verify metadata is preserved
sample = text_chunks[0]
print(f"\nSample chunk metadata: {sample.metadata}")
print(f"Sample content (first 200 chars): {sample.page_content[:200]}")

## Step 4: Embeddings

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

def get_embeddings(model_name: str = "sentence-transformers/all-MiniLM-L6-v2"):
    return HuggingFaceEmbeddings(model_name=model_name)

embedding_model = get_embeddings()

# Sanity check
test_vec = embedding_model.embed_query("diabetes symptoms")
print(f"✅ Embedding dimension: {len(test_vec)}")

## Step 5: Pinecone Vector Store (Single Index, Two Namespaces)

We use **two namespaces** within the same index:
- `medical-general` → disease + herbal chunks (for symptom/disease answering)
- `medical-nutrition` → nutrition chunks only (for dietary recommendations)

In [ ]:
import time
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore

pc         = Pinecone(api_key=PINECONE_API_KEY)
INDEX_NAME = "medical-chatbot"

# ── Create index if it doesn't exist ──
existing = pc.list_indexes().names()
if INDEX_NAME not in existing:
    print(f"Creating index: {INDEX_NAME}")
    pc.create_index(
        name=INDEX_NAME,
        dimension=384,           # all-MiniLM-L6-v2 dimension
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )
    while not pc.describe_index(INDEX_NAME).status["ready"]:
        time.sleep(1)
    print("✅ Index ready")
else:
    print(f"✅ Using existing index: {INDEX_NAME}")

# ── Separate chunks by book type ──
disease_herbal_chunks = [c for c in text_chunks if c.metadata.get("book_type") in ("disease", "herbal")]
nutrition_chunks      = [c for c in text_chunks if c.metadata.get("book_type") == "nutrition"]

print(f"\nDisease+Herbal chunks : {len(disease_herbal_chunks)}")
print(f"Nutrition chunks      : {len(nutrition_chunks)}")

In [ ]:
# ── Upsert disease+herbal into namespace 'general' ──
# (Skip this cell if index is already populated — use Step 5b instead)

docsearch_general = PineconeVectorStore.from_documents(
    documents=disease_herbal_chunks,
    embedding=embedding_model,
    index_name=INDEX_NAME,
    namespace="general"
)
print("✅ Disease + Herbal chunks upserted into namespace 'general'")

# ── Upsert nutrition into namespace 'nutrition' ──
docsearch_nutrition = PineconeVectorStore.from_documents(
    documents=nutrition_chunks,
    embedding=embedding_model,
    index_name=INDEX_NAME,
    namespace="nutrition"
)
print("✅ Nutrition chunks upserted into namespace 'nutrition'")

In [ ]:
# ── Step 5b: Load existing index (run this instead of 5a on subsequent sessions) ──

docsearch_general = PineconeVectorStore.from_existing_index(
    index_name=INDEX_NAME,
    embedding=embedding_model,
    namespace="general"
)

docsearch_nutrition = PineconeVectorStore.from_existing_index(
    index_name=INDEX_NAME,
    embedding=embedding_model,
    namespace="nutrition"
)

print("✅ Both namespaces loaded from existing index")

## Step 6: LLM Setup (Groq)

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=GROQ_API_KEY,
    temperature=0,
    max_retries=2,
)

print("✅ LLM ready")

## Step 7: Pipeline Architecture

```
User Query
    |
    v  Step 0 -- _extract_symptoms()
         LLM -> JSON { symptoms, duration, temp, age, worsening, ... }
    |
    v  Step 1 -- disease_chain (RAG, namespace: general)
         symptom_context + retrieved chunks -> Llama-3.3-70B
         -> IDENTIFIED_DISEASE: <name>
    |
    +-- if result = 'general' --> Step 2 -- fallback_disease_chain
    |                                       Pure LLM, no Pinecone
    v
    Step 3 -- nutrition_chain (RAG, namespace: nutrition)
         Richer query: 'disease + nutritional requirements dietary guidelines'
         + disease_info from Step 1/2 -> 7-row table (up/down vs normal)
    |
    v  Step 4 -- assess_severity()
         Structured rules: temp>=104F->URGENT, >=105F->CRITICAL
                           age<2 or >70 -> minimum MODERATE
                           7+ days + worsening -> URGENT
         Keyword match: CRITICAL_CONDITIONS, URGENT_SYMPTOMS sets
         LLM triage: balanced prompt with calibration rules
         Final = worst of all three (MILD diseases capped at MODERATE)
```

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from operator import itemgetter
import re, json

# ─── STEP 0: Symptom Extraction ─────────────────────────────────────────
# Runs before any chain. Parses raw query -> structured JSON.
# ─────────────────────────────────────────────────────────────────────────
SYMPTOM_EXTRACTION_PROMPT = """\
You are a clinical intake assistant. Extract structured information from the
patient's message below. Respond ONLY in this exact JSON format with no extra text:

{{
  "symptoms": ["symptom1", "symptom2"],
  "duration": "e.g. 3 days / since morning / unknown",
  "severity_words": ["e.g. severe / mild / moderate"],
  "temperature": "e.g. 100 F / 38 C / not mentioned",
  "location": "e.g. chest / head / abdomen / whole body / not mentioned",
  "onset": "e.g. sudden / gradual / unknown",
  "age": "e.g. 25 / child / elderly / not mentioned",
  "existing_conditions": ["e.g. diabetes / none mentioned"],
  "worsening": "yes / no / unknown"
}}

Patient message: {query}"""

symptom_extraction_chain = (
    ChatPromptTemplate.from_messages([
        ('system', SYMPTOM_EXTRACTION_PROMPT),
        ('human',  '{query}')
    ])
    | llm
    | StrOutputParser()
)

def _extract_symptoms(query: str) -> dict:
    """Step 0: parse raw query into structured JSON. Returns safe defaults on failure."""
    try:
        raw   = symptom_extraction_chain.invoke({'query': query})
        clean = re.sub(r'```(?:json)?|```', '', raw).strip()
        return json.loads(clean)
    except Exception as e:
        print(f"Symptom extraction failed: {e}")
        return {"symptoms":[],"duration":"unknown","severity_words":[],
                "temperature":"not mentioned","location":"not mentioned",
                "onset":"unknown","age":"not mentioned",
                "existing_conditions":[],"worsening":"unknown"}

def _format_symptom_context(sx: dict) -> str:
    return '\n'.join([
        f"Symptoms     : {', '.join(sx.get('symptoms',[])) or 'not specified'}",
        f"Duration     : {sx.get('duration','unknown')}",
        f"Temperature  : {sx.get('temperature','not mentioned')}",
        f"Severity     : {', '.join(sx.get('severity_words',[])) or 'not specified'}",
        f"Location     : {sx.get('location','not mentioned')}",
        f"Onset        : {sx.get('onset','unknown')}",
        f"Age          : {sx.get('age','not mentioned')}",
        f"Conditions   : {', '.join(sx.get('existing_conditions',[])) or 'none mentioned'}",
        f"Worsening    : {sx.get('worsening','unknown')}",
    ])

# ─── CHAIN A: Disease Identification (RAG) ───────────────────────────────
# Now receives symptom_context alongside retrieved Pinecone chunks.
# ─────────────────────────────────────────────────────────────────────────
DISEASE_SYSTEM_PROMPT = """\
You are a medical assistant for preliminary disease identification.
Use the retrieved medical context AND the structured symptom summary below.

STRUCTURED PATIENT CONTEXT:
{symptom_context}

DISEASE IDENTIFICATION RULES:
1. Use the structured context above as your PRIMARY signal.
2. Match symptom clusters confidently to the most likely condition.
3. Common patterns (use as a guide, not a limit):
   - Fever + headache + chills + body ache -> Influenza / Viral Fever
   - Fever + sore throat + cough -> URTI / Pharyngitis
   - Frequent urination + thirst + fatigue -> Diabetes Mellitus
   - Chest pain + breathlessness + sweating -> Cardiac condition
   - Fatigue + pale skin + dizziness -> Anemia
   - High BP + headache -> Hypertension
   - Abdominal pain + nausea + loose stool -> Gastroenteritis
   - Joint pain + swelling + stiffness -> Arthritis
   - Skin rash + itching -> Dermatitis / Allergic Reaction
   - Cough + mucus + mild fever -> Bronchitis / URTI
   - Burning urination + frequency -> UTI
   - Yellowing skin + dark urine + fatigue -> Jaundice / Hepatitis
   - Sudden weight loss + fatigue + night sweats -> Tuberculosis / Lymphoma
4. Duration matters -- symptoms for 3+ days narrow toward specific infections.
5. Adjust for age: children -> viral illnesses; elderly -> pneumonia/cardiac.
6. ONLY use 'general' if symptoms are completely contradictory or meaningless.
7. Keep the clinical explanation to 3-5 sentences.
8. Do NOT add dietary advice -- only explain the condition and its mechanism.

At the end of your answer on a NEW LINE output exactly:
IDENTIFIED_DISEASE: <disease name>
If no specific disease can be determined write:
IDENTIFIED_DISEASE: general

Retrieved medical context:
{context}"""

retriever_general = docsearch_general.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 4}
)

disease_chain = (
    {
        'context':         itemgetter('input') | retriever_general,
        'input':           itemgetter('input'),
        'symptom_context': itemgetter('symptom_context'),
    }
    | ChatPromptTemplate.from_messages([
        ('system', DISEASE_SYSTEM_PROMPT),
        ('human',  '{input}')
    ])
    | llm
    | StrOutputParser()
)

# ─── CHAIN A FALLBACK: Pure LLM (no RAG) ────────────────────────────────
# Fires only when Chain A returns 'general' / 'unknown' / ''
# ─────────────────────────────────────────────────────────────────────────
FALLBACK_DISEASE_PROMPT = """\
You are an experienced physician doing a preliminary assessment.
The patient describes the following symptoms:

{query}

Structured symptom analysis:
{symptom_context}

Based purely on your medical knowledge (no retrieved context needed):
1. What is the SINGLE most likely preliminary diagnosis?
2. Give a 3-4 sentence clinical explanation.
3. Mention 1-2 differential diagnoses briefly.

Be confident -- a preliminary assessment with partial information is
better than saying 'I don't know'. A real doctor would give their best
clinical impression.

End on a NEW LINE with exactly:
IDENTIFIED_DISEASE: <most likely condition name>"""

fallback_disease_chain = (
    ChatPromptTemplate.from_messages([
        ('system', FALLBACK_DISEASE_PROMPT),
        ('human',  'Please give your best clinical assessment.')
    ])
    | llm
    | StrOutputParser()
)

def _extract_disease(text: str) -> str:
    m = re.search(r'IDENTIFIED_DISEASE:\s*(.+)', text, re.IGNORECASE)
    return m.group(1).strip() if m else 'general'

print('Chain A (RAG) + Chain A fallback + symptom extractor ready')

In [ ]:
# ─── CHAIN B: Nutrition Recommendation (RAG) ───────────────────────────
# Two upgrades vs original:
#  1. Richer retrieval query for better Pinecone results
#  2. Receives disease_info so it can explain WHY each value is up/down
# ─────────────────────────────────────────────────────────────────────────

NUTRITION_SYSTEM_PROMPT = """\
You are a senior clinical dietitian. A patient has been diagnosed with: {disease}

Clinical context about this condition:
{disease_info}

Using ONLY the retrieved nutrition knowledge below AND your clinical expertise
for this SPECIFIC disease, output a personalised daily intake table.

STRICT RULES:
1. Values MUST be specific to {disease} -- do NOT use generic healthy-adult defaults.
2. For each nutrient, actively decide: should it be HIGHER, LOWER, or RESTRICTED
   compared to a healthy adult? Reflect that in the % and the note.
3. If a nutrient is contraindicated or must be severely restricted for {disease},
   write the restriction clearly (e.g. 'Restrict to <10%' or 'Limit to 1.5 L/day').
4. The Notes column MUST explain WHY -- what does this disease do that changes the need?
5. Output ONLY the markdown table below. No bullet points, no extra text.

| Nutrient      | Daily Recommended (for {disease}) | Notes -- Why this amount?   |
|---------------|-----------------------------------|---------------------------------|
| Carbohydrates | XX% (up/down vs normal)           | ...                             |
| Protein       | XX% (up/down vs normal)           | ...                             |
| Fat           | XX% (up/down vs normal)           | ...                             |
| Vitamins      | Key vitamins + amounts            | ...                             |
| Minerals      | Key minerals + amounts            | ...                             |
| Water         | X.X L/day (up/down vs normal)    | ...                             |
| Fiber         | XX g/day (up/down vs normal)     | ...                             |

Retrieved nutrition knowledge:
{context}"""

def _nutrition_query(inputs: dict) -> str:
    """Richer query gives far better Pinecone retrieval than just disease name."""
    return f"{inputs['disease']} nutritional requirements dietary guidelines management"

retriever_nutrition = docsearch_nutrition.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 5}
)

nutrition_chain = (
    {
        'context':      _nutrition_query | retriever_nutrition,
        'disease':      itemgetter('disease'),
        'disease_info': itemgetter('disease_info'),  # new: clinical context
    }
    | ChatPromptTemplate.from_messages([
        ('system', NUTRITION_SYSTEM_PROMPT),
        ('human',  'Patient disease: {disease}\n\nProvide the personalised daily nutritional requirements.')
    ])
    | llm
    | StrOutputParser()
)

print('Nutrition Chain ready (richer retrieval query + disease_info)')

## Step 8: Combined Pipeline Function

In [ ]:
# ─── SEVERITY TRIAGE ────────────────────────────────────────────────────
# Upgraded: now receives symptom_data dict with parsed age, temp, duration.
# 3 inputs -> final = worst of structured rules, keyword match, LLM.
# ─────────────────────────────────────────────────────────────────────────
import re

CRITICAL_CONDITIONS = {
    'heart attack', 'myocardial infarction', 'cardiac arrest', 'heart failure',
    'stroke', 'brain hemorrhage', 'intracranial hemorrhage', 'status epilepticus',
    'meningitis', 'encephalitis', 'pulmonary embolism', 'respiratory failure',
    'anaphylaxis', 'anaphylactic shock', 'sepsis', 'septic shock',
    'toxic shock syndrome', 'bowel perforation', 'aortic aneurysm',
    'aortic dissection', 'ruptured ectopic', 'eclampsia',
    'diabetic ketoacidosis', 'dka', 'hypoglycemic coma',
    'internal bleeding', 'severe hemorrhage', 'hypertensive crisis',
}

URGENT_SYMPTOMS = {
    'chest pain', 'chest tightness', 'chest pressure', 'crushing chest',
    'pain radiating to arm', 'pain radiating to jaw',
    "can't breathe", 'cannot breathe', 'unable to breathe',
    'difficulty breathing', 'coughing blood', 'blood in sputum',
    'sudden severe headache', 'worst headache of my life',
    'sudden confusion', 'slurred speech', 'face drooping', 'arm weakness',
    'sudden vision loss', 'loss of consciousness', 'unconscious',
    'seizure', 'convulsion',
    'vomiting blood', 'blood in stool', 'rectal bleeding',
    'severe bleeding', 'coughing up blood',
    'throat swelling', 'tongue swelling',
    'fever above 104', 'fever 105', 'fever 106',
    'bluish lips', 'blue lips',
    'overdose', 'poisoned', 'not breathing', 'baby not moving',
}

# Known mild conditions -- always cap at MODERATE regardless of LLM output
MILD_CONDITIONS = {
    'common cold', 'influenza', 'viral fever', 'flu',
    'upper respiratory tract infection', 'urti',
    'mild fever', 'low grade fever',
    'indigestion', 'acid reflux', 'heartburn',
    'mild headache', 'tension headache',
    'mild gastroenteritis', 'stomach bug',
    'mild allergic reaction', 'hay fever',
    'minor cut', 'bruise', 'sprain',
    'mild anemia', 'fatigue',
}

def assess_severity(query: str, disease: str, disease_info: str,
                    symptom_data: dict = None) -> dict:
    """
    3-step severity with structured symptom context.
    symptom_data: dict from _extract_symptoms() -- age, temp, duration, worsening.
    Returns a dict with severity level + full alert payload.
    """
    sx            = symptom_data or {}
    query_lower   = query.lower()
    disease_lower = disease.lower()
    info_lower    = disease_info.lower()

    # Parse structured fields
    duration    = sx.get('duration', 'unknown').lower()
    age_raw     = sx.get('age', 'not mentioned').lower()
    worsening   = sx.get('worsening', 'unknown').lower()
    temperature = sx.get('temperature', 'not mentioned').lower()

    age_num = None
    am = re.search(r'\\b(\\d+)\\b', age_raw)
    if am: age_num = int(am.group(1))

    temp_val = None
    tf = re.search(r'(\\d+\\.?\\d*)\\s*[°]?\\s*[fF]', temperature)
    if tf: temp_val = float(tf.group(1))
    tc = re.search(r'(\\d+\\.?\\d*)\\s*[°]?\\s*[cC]', temperature)
    if tc: temp_val = float(tc.group(1)) * 9/5 + 32

    dur_days = None
    dm = re.search(r'(\\d+)\\s*day', duration)
    if dm: dur_days = int(dm.group(1))

    # Step 1: Structured pre-checks
    structured = None
    if temp_val is not None:
        if   temp_val >= 105: structured = 'CRITICAL'
        elif temp_val >= 104: structured = 'URGENT'
        elif temp_val >= 103 and age_num is not None and age_num < 2:
            structured = 'URGENT'
    if dur_days and dur_days >= 7 and worsening == 'yes' and not structured:
        structured = 'URGENT'
    if age_num is not None and not structured:
        if age_num < 2 or age_num > 70: structured = 'MODERATE'

    # Step 2: Keyword match
    instant = None
    crit_m = next((c for c in CRITICAL_CONDITIONS
                   if c in disease_lower or c in info_lower), None)
    if not crit_m:
        crit_q = {'heart attack','stroke','overdose','not breathing',
                  'poisoned','anaphylaxis','seizure','convulsion'}
        crit_m = next((c for c in crit_q if c in query_lower), None)
    urg_m = next((s for s in URGENT_SYMPTOMS if s in query_lower), None)
    if crit_m: instant = 'CRITICAL'
    elif urg_m: instant = 'URGENT'

    # Step 3: LLM triage
    sev_prompt = ChatPromptTemplate.from_messages([
        ('system',
         'You are a senior emergency physician doing triage.\n'
         'Classify severity as CRITICAL, URGENT, MODERATE, or MILD.\n\n'
         'CRITICAL -- life-threatening: heart attack, stroke, anaphylaxis, sepsis, overdose.\n'
         'URGENT   -- needs doctor today: fever 103F+, worsening infection, fracture.\n'
         'MODERATE -- doctor in 2-3 days: mild infection, UTI, rash, stable flu.\n'
         'MILD     -- manage at home: cold, indigestion, 1-2 day fever <103F.\n\n'
         'CONTEXT: Duration: {duration} | Age: {age} | Worsening: {worsening} | Temp: {temperature}\n\n'
         'CALIBRATION: Flu/cold -> MILD or MODERATE only.\n'
         'When unsure MILD vs MODERATE -> MODERATE.\n'
         'When unsure MODERATE vs URGENT -> MODERATE.\n'
         'CRITICAL only if objectively life-threatening.\n\n'
         'Respond EXACTLY:\nSEVERITY: <level>\nREASON: <one sentence>\nACTION: <one sentence>'),
        ('human', 'Disease: {disease}\nDescription: {disease_info}\nQuery: {query}')
    ])
    try:
        resp = (sev_prompt | llm | StrOutputParser()).invoke({
            'disease': disease, 'disease_info': disease_info[:600],
            'query': query[:300], 'duration': duration,
            'age': age_raw, 'worsening': worsening, 'temperature': temperature,
        })
        sm = re.search(r'SEVERITY:\s*(CRITICAL|URGENT|MODERATE|MILD)', resp, re.I)
        rm = re.search(r'REASON:\s*(.+)',  resp, re.I)
        am2= re.search(r'ACTION:\s*(.+)',  resp, re.I)
        llm_level  = sm.group(1).upper() if sm else 'MODERATE'
        llm_reason = rm.group(1).strip() if rm else ''
        llm_action = am2.group(1).strip() if am2 else ''
    except:
        llm_level, llm_reason, llm_action = 'MODERATE', '', ''

    # Final: worst of three; MILD diseases capped at MODERATE
    order = {'CRITICAL':4,'URGENT':3,'MODERATE':2,'MILD':1}
    if any(m in disease_lower for m in MILD_CONDITIONS):
        instant   = None
        llm_level = min(llm_level, 'MODERATE', key=lambda x: order.get(x,0))

    candidates = [llm_level]
    if instant:    candidates.append(instant)
    if structured: candidates.append(structured)
    final = max(candidates, key=lambda x: order.get(x,0))

    alert_cfg = {
        'CRITICAL': {'show':True,  'color':'#ff1744','icon':'CRITICAL',
                     'title':'EMERGENCY -- Go to Hospital Immediately!', 'call':'108'},
        'URGENT':   {'show':True,  'color':'#ff6d00','icon':'URGENT',
                     'title':'See a Doctor Today -- Do Not Delay',       'call':None},
        'MODERATE': {'show':False, 'color':'#ffd600','icon':'MODERATE',
                     'title':'Doctor Visit Recommended',                 'call':None},
        'MILD':     {'show':False, 'color':'#00c853','icon':'MILD',
                     'title':'Manageable at Home',                       'call':None},
    }
    cfg = alert_cfg[final]
    return {
        'severity': final, 'show_alert': cfg['show'],
        'alert_color': cfg['color'], 'alert_icon': cfg['icon'],
        'alert_title': cfg['title'],
        'alert_message': llm_action or cfg['title'],
        'emergency_call': cfg['call'], 'reason': llm_reason,
    }


# ─── FULL PIPELINE ───────────────────────────────────────────────────────
def medical_nutrition_chatbot(query: str, verbose: bool = True) -> dict:
    """
    Step 0 -- Symptom extraction  -> structured JSON
    Step 1 -- RAG disease chain   -> IDENTIFIED_DISEASE
    Step 2 -- Fallback chain      -> fires only if Step 1 = general
    Step 3 -- Nutrition chain     -> 7-row up/down table
    Step 4 -- Severity triage     -> structured rules + keyword + LLM
    """
    if verbose: print(f"\n{'='*60}\nQuery: {query}\n{'='*60}")

    # Step 0
    if verbose: print('Step 0: Extracting structured symptoms...')
    sx          = _extract_symptoms(query)
    symptom_ctx = _format_symptom_context(sx)
    if verbose: print(f"  {sx.get('symptoms')} | {sx.get('duration')} | {sx.get('temperature')}")

    # Step 1 -- RAG
    if verbose: print('Step 1: RAG disease identification...')
    disease_raw        = disease_chain.invoke({'input': query, 'symptom_context': symptom_ctx})
    identified_disease = _extract_disease(disease_raw)
    clean_disease      = re.sub(r'\nIDENTIFIED_DISEASE:.*', '', disease_raw, flags=re.I).strip()
    if verbose: print(f'  RAG identified: {identified_disease}')

    # Step 2 -- Fallback
    used_fallback = False
    if identified_disease.lower() in ('general', 'unknown', ''):
        if verbose: print('Step 2: RAG returned general -- running LLM fallback...')
        try:
            fb_raw     = fallback_disease_chain.invoke({'query': query, 'symptom_context': symptom_ctx})
            fb_disease = _extract_disease(fb_raw)
            fb_clean   = re.sub(r'\nIDENTIFIED_DISEASE:.*', '', fb_raw, flags=re.I).strip()
            if fb_disease.lower() not in ('general', 'unknown', ''):
                identified_disease = fb_disease
                clean_disease      = fb_clean
                used_fallback      = True
                if verbose: print(f'  Fallback identified: {identified_disease}')
            else:
                if verbose: print('  Fallback also returned general')
        except Exception as e:
            if verbose: print(f'  Fallback error: {e}')

    # Step 3 -- Nutrition
    if identified_disease.lower() in ('general', 'unknown', ''):
        nutrition_response = (
            'I need a bit more information to give you an accurate nutrition plan. '
            'Could you share: how long you have had these symptoms, your age, '
            'and any existing health conditions or medications?'
        )
    else:
        if verbose: print(f'Step 3: Nutrition for {identified_disease}...')
        nutrition_response = nutrition_chain.invoke({
            'disease':      identified_disease,
            'disease_info': clean_disease[:600],
        })

    # Step 4 -- Severity
    if verbose: print('Step 4: Severity assessment...')
    severity = assess_severity(query, identified_disease, clean_disease, sx)
    if verbose: print(f"  Severity: {severity['severity']}")

    return {
        'query':              query,
        'identified_disease': identified_disease,
        'disease_info':       clean_disease,
        'nutrition_plan':     nutrition_response,
        'severity':           severity,
        'used_fallback':      used_fallback,
    }


def format_full_response(result: dict) -> str:
    sev = result['severity']
    sep = '=' * 70
    return (
        f'\n{sep}\n'
        f'MEDICAL CHATBOT RESPONSE\n'
        f'{sep}\n\n'
        f"QUERY: {result['query']}\n\n"
        f"{'─'*70}\n"
        f"DISEASE:   {result['identified_disease'].upper()}\n"
        f"           (via fallback: {result['used_fallback']})\n"
        f"SEVERITY:  {sev['severity']} -- {sev['reason']}\n"
        f"           Action: {sev['alert_message']}\n"
        f"{'─'*70}\n\n"
        f"{result['disease_info']}\n\n"
        f"{'─'*70}\n"
        f'RECOMMENDED DAILY NUTRITIONAL INTAKE\n'
        f"{'─'*70}\n\n"
        f"{result['nutrition_plan']}\n"
        f'{sep}'
    )

print('Pipeline + severity + format functions defined')

## Step 9: Run the Chatbot -- Example Queries

In [ ]:
# Example 1: Influenza (was returning 'general' before the upgrade)
result = medical_nutrition_chatbot(
    "Hello, I had a headache for 3 days, fever around 100 F last night, "
    "and my body is shaking due to chills. What disease is this?"
)
print(format_full_response(result))

In [ ]:
# ── Example 2: Hypertension ──
result = medical_nutrition_chatbot(
    "I suffer from high blood pressure and occasional headaches. "
    "What nutritional changes should I make?"
)
print(format_full_response(result))

In [ ]:
# ── Example 3: Anemia ──
result = medical_nutrition_chatbot(
    "I feel constantly tired, pale skin, shortness of breath and dizziness."
)
print(format_full_response(result))

In [ ]:
# ── Example 4: Custom user input ──
user_input = input("Describe your symptoms: ")
result = medical_nutrition_chatbot(user_input)
print(format_full_response(result))

## Step 10: Interactive Session Loop (Optional)

Run this cell for a continuous conversation. Type `exit` to quit.

In [ ]:
print("🏥 Medical Nutrition Chatbot — type 'exit' to quit\n")

while True:
    query = input("\nDescribe your symptoms or ask a question: ").strip()
    if not query:
        continue
    if query.lower() in ("exit", "quit", "q"):
        print("👋 Session ended.")
        break
    result = medical_nutrition_chatbot(query, verbose=False)
    print(format_full_response(result))

---
## Architecture Summary

| Component | Detail |
|---|---|
| Embeddings | `sentence-transformers/all-MiniLM-L6-v2` (384-dim) |
| Vector DB | Pinecone -- 2 namespaces: `general`, `nutrition` |
| LLM | Groq `llama-3.3-70b-versatile` |
| Step 0 | `_extract_symptoms()` -- LLM parses query -> structured JSON |
| Chain A | RAG disease chain -- uses `symptom_context` + retrieved chunks |
| Chain A fallback | `fallback_disease_chain` -- pure LLM (fires when RAG returns `general`) |
| Chain B | Nutrition chain -- richer retrieval query + `disease_info` -> up/down table |
| Severity | Structured rules (age/temp/duration) + keyword match + LLM -> worst wins |
| MILD cap | Known mild diseases capped at MODERATE regardless of LLM output |
| Books | `medic_book.pdf`, `Herbal_Nutrients_and_Their_Health_Benefits.pdf`, `nutrition.pdf` |

### Nutrition up/down Symbols

| Symbol | Meaning | Example |
|---|---|---|
| `up` | Higher than healthy adult baseline | Protein up for TB -- tissue repair |
| `down` | Lower than healthy adult baseline | Carbs down for Diabetes -- glucose control |
| `Restrict` | Severely limited | Potassium for CKD |
| (none) | At normal level | Fiber for most infections |

Healthy adult baseline: Carbs 45-65%, Protein 10-35%, Fat 20-35%, Water 2.7 L/day, Fiber 25-30 g/day.

### What changed vs original notebook

| What | Before | After |
|---|---|---|
| Step 0 | Not present | `_extract_symptoms()` parses query -> structured JSON |
| Chain A input | Raw query only | Raw query + `symptom_context` |
| Chain A fallback | Not present | `fallback_disease_chain` (pure LLM, no RAG) |
| Chain B retrieval | Disease name only | Richer query: `disease + nutritional requirements dietary guidelines` |
| Chain B prompt | Generic table | Receives `disease_info`; forces up/down notation with WHY column |
| Severity input | query + disease + disease_info | + structured `symptom_data` (age, temp, duration, worsening) |
| Severity rules | Keyword match -> LLM | Structured pre-checks -> keyword match -> LLM |
| MILD_CONDITIONS | Not present | Known mild diseases capped at MODERATE |
| LLM severity prompt | 'when in doubt choose CRITICAL' | Rebalanced with MILD/MODERATE calibration rules |

> Disclaimer: This chatbot is for pre-screening only. Always consult a qualified healthcare professional.